# CSLR Project Script Runner
This notebook locates and executes project scripts in a controlled, logged sequence. It captures outputs, execution times, and failures for a concise run report.

## 1. Individua gli script e configura i percorsi
Definisci le directory di lavoro e raccogli la lista degli script da eseguire (es. .py), con filtri e ordinamento.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import List

# Project root can be adjusted if needed.
PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "phoenix_project"

# Collect Python scripts, excluding private and test-like files.
script_paths: List[Path] = []
if SCRIPTS_DIR.exists():
    script_paths = sorted(
        p
        for p in SCRIPTS_DIR.rglob("*.py")
        if not p.name.startswith("_") and "test" not in p.name.lower()
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Scripts dir:   {SCRIPTS_DIR}")
print(f"Scripts found: {len(script_paths)}")
for p in script_paths:
    print(f" - {p.relative_to(PROJECT_ROOT)}")

## 2. Crea un runner che esegue tutti gli script
Implementa una funzione che invoca gli script tramite subprocess, impostando working directory e variabili d'ambiente.

In [ ]:
from __future__ import annotations

import subprocess
import sys
import time
from dataclasses import dataclass
from typing import Dict, Optional


@dataclass
class RunResult:
    script: Path
    return_code: int
    duration_sec: float
    stdout: str
    stderr: str


def run_script(path: Path, working_dir: Path, env: Optional[Dict[str, str]] = None) -> RunResult:
    """Execute a Python script with subprocess and capture outputs.

    Args:
        path: Script path.
        working_dir: Working directory for execution.
        env: Optional environment overrides.

    Returns:
        RunResult with execution metadata.
    """
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    start = time.perf_counter()
    completed = subprocess.run(
        [sys.executable, str(path)],
        cwd=str(working_dir),
        env=merged_env,
        capture_output=True,
        text=True,
    )
    duration = time.perf_counter() - start
    return RunResult(
        script=path,
        return_code=completed.returncode,
        duration_sec=duration,
        stdout=completed.stdout,
        stderr=completed.stderr,
    )

## 3. Esegui in sequenza e cattura output
Esegui gli script in ordine, catturando stdout/stderr e tempi di esecuzione, salvando i risultati in una struttura dati.

In [ ]:
from __future__ import annotations

from typing import List

results: List[RunResult] = []

for script_path in script_paths:
    print(f"Running: {script_path.relative_to(PROJECT_ROOT)}")
    result = run_script(script_path, working_dir=PROJECT_ROOT)
    results.append(result)
    print(f"  Return code: {result.return_code}")
    print(f"  Duration:    {result.duration_sec:.2f}s")

## 4. Report di esecuzione e gestione errori
Aggrega lo stato di esecuzione, gestisci errori/exit code, e mostra un report finale con eventuali fallimenti.

In [ ]:
failed = [r for r in results if r.return_code != 0]

print("\n=== Execution Report ===")
print(f"Total scripts: {len(results)}")
print(f"Succeeded:     {len(results) - len(failed)}")
print(f"Failed:        {len(failed)}")

if failed:
    print("\nFailures:")
    for r in failed:
        print(f"- {r.script.relative_to(PROJECT_ROOT)}")
        print(f"  Code: {r.return_code}")
        if r.stderr.strip():
            print("  Stderr (last 10 lines):")
            lines = r.stderr.strip().splitlines()[-10:]
            for line in lines:
                print(f"    {line}")
else:
    print("All scripts completed successfully.")